## Load search library
Provide either a path to a `.csv`/`.tsv` file (new library) or a `.parquet` file (a previously processed and saved library). Replace the current path with the location of your downloaded or saved library.

Processing a new `.csv`/`.tsv` library takes a little time, since each compound goes through the following steps:
1. SMILES are reduced to their largest fragment and canonicalized.
2. Exact mass, protonated mass, and sodiated mass are calculated.
3. A boolean flag is set if the compound contains an odd number of nitrogens.
4. Functional groups are assigned based on SMARTS patterns.

Once processing is complete, the result is automatically saved as a `.parquet` file with the same name as the source file (e.g. `coconut_csv-05-2026.csv` → `coconut_csv-05-2026.parquet`) in the same directory. On future runs, point the path to that `.parquet` file instead to skip reprocessing.

**Note:** The cache is not automatically invalidated — if you edit or replace the source `.csv`/`.tsv` file, the notebook will always overwrite the old `.parquet` file when reprocessing, but if you instead point the path directly at an existing `.parquet` file, you'll get whatever was cached, even if the source has since changed. Delete the `.parquet` file or point to a new path if you want to force reprocessing.

In [ ]:
from smiles_library import load_smiles_library

# replace the path to the libray to match your environment
library_df  = load_smiles_library("../stored/coconut_csv-05-2026.csv")

### Load data for training
1. CSV file with SMILES and bruker filenames
2. Root directory (if used) of bruker .0 files

SMILES are reduced to largest fragment and canonicalized

Spectra are transformed to absorbance, baseline corrected, and normalized

Expected 496 spectral files

In [1]:
from ir_processing import TrainingData

# since using OOB for F1 score calculation, there will be small variations
# in final model results based on the order of the training data. Order
# is dictated by the csv file
training_data = TrainingData(
                    csv_path  = "../training_data/smiles_to_file_V2.csv",
                    spec_root = "../training_data",
                )

print(f"Records imported for training: {training_data.size()}")

Records imported for training: 496


### Automatic processing of training data

Data is shaped for input to the classification model. 

Each functional group is default sampled with 16 points to normalize vector lengths.

Functional groups lacking sufficient representation (< 5 instances) are masked.


In [2]:
from modeling import SpectIRmodeling

# if run without the training_data input, the model will try
# to initialize from a previously saved training in the ../stored directory
cls_model = SpectIRmodeling(training_data, model_params={ "n_estimators": 200, "max_depth": 24, "min_samples_leaf": 3 })

INFO: Training data provided. Initializing training pipeline...
INFO: Successfully saved model and metadata to '../stored/rf_multioutput_model.joblib'!


### Model training

Expected training shape:
    X_regional shape: (496, 848)
    Y_filtered shape: (496, 37)
      
Default training has Out-Of-Bag flag set to True to calculate precision, recall and F1

Expected results (precision, recall, F1):
1. OVERALL (Samples): 0.748  0.855  0.784           
2. OVERALL  (Macro) : 0.602  0.581  0.573           

In [7]:
cls_model.set_threshold(0.35)
print(cls_model.summary)

Final data shape: X=(496, 848), Y=(496, 37)
OOB Metrics (Per-Compound / Samples): Precision=0.714, Recall=0.887, F1-Score=0.777
OOB Metrics (Per-Class / Weighted):   Precision=0.711, Recall=0.882, F1-Score=0.781


In [8]:
print(cls_model.metrics)

                  FG Label  Support  Precision  Recall  F1-Score Confidence
12        alkyl_CH_stretch      463      0.933   1.000     0.966   complete
33            CH2_CH3_bend      462      0.931   1.000     0.965   complete
11             aromatic_CH      329      0.803   0.964     0.876       high
31         aromatic_CH_oop      329      0.803   0.964     0.876       high
22             aromatic_CC      290      0.749   0.976     0.847       high
1               alcohol_OH      351      0.736   0.983     0.841       high
28                ether_CO      314      0.698   0.965     0.810       high
15                ester_CO      161      0.681   0.888     0.771       high
26         ester_CO_single      161      0.681   0.888     0.771       high
23               alkene_CC      286      0.639   0.955     0.766       high
29           aryl_ether_CO      146      0.643   0.863     0.737       high
10               alkene_CH      256      0.586   0.941     0.723       high
32          

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def tune_thresholds(model: SpectIRmodeling, threshold: float):
    n_samples, n_outputs = model.Y.shape
    y_pred_oob = np.zeros((n_samples, n_outputs))

    def max_f1(truth, probs):
        thresholds = [0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
        f1 = []
        for threshold in thresholds:
            pred = (probs >= threshold).astype(int)
            f1.append(f1_score(truth, pred, zero_division=0))
        # print(f1.index(max(f1)))
        return thresholds[f1.index(max(f1))]
    
    # 1. Calculate metrics per functional group
    fg_metrics = []
    for i in range(n_outputs):
        oob_probs = model.model.estimators_[i].oob_decision_function_
        threshold = max_f1(model.Y[:, i], oob_probs[:, 1])
        
        y_pred_oob[:, i] = (oob_probs[:, 1] >= threshold).astype(int)
        
        support = int(model.Y[:, i].sum())
        prec = precision_score(model.Y[:, i], y_pred_oob[:, i], zero_division=0)
        rec = recall_score(model.Y[:, i], y_pred_oob[:, i], zero_division=0)
        f1 = f1_score(model.Y[:, i], y_pred_oob[:, i], zero_division=0)

        # Baseline random chance prevalence
        prevalence = support / float(n_samples) if n_samples > 0 else 0

        # Determine the confidence tier
        if support >= 30 and prec >= 0.90 and rec >= 0.80:
            confidence = "complete"
        elif support >= 40 and f1 >= 0.70:
            confidence = "high"
        elif support >= 15 and (f1 >= 0.40 or f1 > (prevalence * 5)):
            confidence = "medium"
        else:
            confidence = "low"

        fg_metrics.append({
            "FG Label": model.labels[i],
            "Support": support,
            "Precision": round(prec, 3),
            "Recall": round(rec, 3),
            "F1-Score": round(f1, 3),
            "Confidence": confidence,
            "Threshold": threshold,
            # "type": 0,
        })

    # 2. Calculate the OVERALL Macro-Average across all valid functional groups
    overall_precision = precision_score(model.Y, y_pred_oob, average='samples', zero_division=0)
    overall_recall = recall_score(model.Y, y_pred_oob, average='samples', zero_division=0)
    overall_f1 = f1_score(model.Y, y_pred_oob, average='samples', zero_division=0)

    # 3. Append overall metrics
    total_support = int(model.Y.sum())
    fg_metrics.append({
        "FG Label": "OVERALL (Samples Avg)",
        "Support": total_support,
        "Precision": round(overall_precision, 3),
        "Recall": round(overall_recall, 3),
        "F1-Score": round(overall_f1, 3),
        "Confidence": "",
        "Threshold": round(sum([metric['Threshold'] for metric in fg_metrics])/len(fg_metrics), 3),
        # "type": 1,
    })
    
    # 2. Calculate the OVERALL Macro-Average across all valid functional groups
    overall_precision = precision_score(model.Y, y_pred_oob, average='macro', zero_division=0)
    overall_recall = recall_score(model.Y, y_pred_oob, average='macro', zero_division=0)
    overall_f1 = f1_score(model.Y, y_pred_oob, average='macro', zero_division=0)

    # 3. Append overall metrics
    total_support = int(model.Y.sum())
    fg_metrics.append({
        "FG Label": "OVERALL (macros Avg)",
        "Support": total_support,
        "Precision": round(overall_precision, 3),
        "Recall": round(overall_recall, 3),
        "F1-Score": round(overall_f1, 3),
        "Confidence": "",
        "Threshold": round(sum([metric['Threshold'] for metric in fg_metrics])/len(fg_metrics), 3),
        # "type": 1,
    })
        
    # 2. Calculate the OVERALL Macro-Average across all valid functional groups
    overall_precision = precision_score(model.Y, y_pred_oob, average='weighted', zero_division=0)
    overall_recall = recall_score(model.Y, y_pred_oob, average='weighted', zero_division=0)
    overall_f1 = f1_score(model.Y, y_pred_oob, average='weighted', zero_division=0)

    # 3. Append overall metrics
    total_support = int(model.Y.sum())
    fg_metrics.append({
        "FG Label": "OVERALL (weighted Avg)",
        "Support": total_support,
        "Precision": round(overall_precision, 3),
        "Recall": round(overall_recall, 3),
        "F1-Score": round(overall_f1, 3),
        "Confidence": "",
        "Threshold": round(sum([metric['Threshold'] for metric in fg_metrics])/len(fg_metrics), 3),
        # "type": 1,
    })
    
    return pd.DataFrame(fg_metrics)

df_tune = tune_thresholds(cls_model, threshold=0.45)

print(df_tune.to_string())

                  FG Label  Support  Precision  Recall  F1-Score Confidence  Threshold
0       carboxylic_acid_OH       38      0.818   0.474     0.600     medium       0.35
1               alcohol_OH      351      0.736   0.983     0.841       high       0.35
2            primary_amine       57      0.529   0.649     0.583     medium       0.30
3       secondary_amine_NH      110      0.664   0.700     0.681     medium       0.40
4       secondary_amide_NH       74      0.662   0.689     0.675     medium       0.35
5         primary_amide_NH       16      0.692   0.562     0.621     medium       0.35
6        aromatic_amine_NH       10      0.250   0.300     0.273        low       0.20
7               pyrrole_NH       40      0.500   0.500     0.500     medium       0.30
8              aldehyde_CH       13      0.333   0.231     0.273        low       0.20
9                alkyne_CH        7      1.000   0.714     0.833        low       0.45
10               alkene_CH      256      0.

## Test and reporting
Use the `find_and_report` function below to analyze and output query spectra. Test data is included in the `test_data` directory. Replace the directory and filename to analyze your own data.

Supported spectrum file formats: `.jdx` (JCAMP-DX) and `.0`.

`cls_model` and `training_data` are created earlier in the notebook — see [relevant cell/section].

In [ ]:
import importlib
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputClassifier

import report
import search
importlib.reload(report)
importlib.reload(search)

# from source_1.modeling import predict_fg_from_spectra
from ir_processing import Spectrum
from search import search_library
from report import build_report_html, plot_spectrum

def find_and_report(
    spectra_dir:  str,                  # directory to data
    spectra_file: str,                  # filename of data
    test_smiles:  str,                  # smiles if known for calculating structural hit index, not required but should be empty string or None
    mw:           float,                # molecular weight for filtering COCONUT results
    model:        SpectIRmodeling,      # model to use for predicting functional groups
    search_id:    str   = None,         # COCONUT id if known, used for exact match hit index
    tanimoto:     float = 0.6,          # Functional group overlap score cuttoff. Only records with scores at or above will be returned
    top_n:        int   = 5000,         # Top number of results to return
    search_mw:    str   = "protonated", # Adduct type to search, defaults to protonated.
    training_data: pd.DataFrame = None, # Reference to training data dataframe for identifying an exact match. Can be useful to verify model.
    group_stereo: bool  = True,         # Yes/no to group stereoisomers. Defaults to yes.
) -> pd.DataFrame:
    
    fpath = spectra_dir + spectra_file
    
    # includes transformation to absorbance, normalize to max peak and baseline correct
    spectrum = Spectrum(fpath, spectra_file, test_smiles)

    # predict the functional groups assigned by random forest model
    predicted_fgs = model.predict_fg_from_spectra(spectrum.wn, spectrum.absorbance)
    
    # plot the search spectra
    fig = plot_spectrum(spectrum.wn, spectrum.absorbance, title=f"third stage spectra: {spectra_file}", smiles='', show_regions=predicted_fgs)
    
    # gather results and counts
    results, mw_count = search_library(
        query_fgs   = predicted_fgs,
        library_df  = library_df,
        mw          = mw,
        top_n       = top_n,
        tanimoto    = tanimoto,
        search_mw   = search_mw,
    )
    
    if group_stereo:
        # group the results by parent coconut identifier. From the COCONUT paper:
        # 'Stereochemical variants of the same molecule are grouped under the same 
        # identifier and are issued a unique postfix to the COCONUT identifier.''
        results = results.groupby(results["parent"]).agg({
            "rank_score":       "mean",
            "fg_set":           "first",
            "smiles":           "first",
            "name":             "first",
            "molecular_weight": "first",
            "fg_len":           "max"
        })

        # add back the identifier and re-sort
        results["id"] = results.index
        results.sort_values(
            by=['rank_score', "fg_len"], 
            inplace=True, 
            ascending=[False, False], 
            ignore_index=True
        )
    
    # generate the html report and open in default browser
    build_report_html(
        spectrum_fig = fig,
        query_fgs    = predicted_fgs,
        candidates   = results,
        query_name   = spectra_file,
        top_n        = 50,
        output_path  = f"../reports/{spectra_file}.html",
        search_stats = {
            "mw_count": mw_count,
            "fg_count": len(results),
        },
        search_param = {
            "mw":        mw,
            "search_mw": search_mw,
            "tanimoto":  tanimoto,
            "test_smi":  test_smiles,
        },
        training_data= training_data.spectra,
        FG_CONFIDENCE= model.confidence,
    )

## Using BondMap
Example implementation of searching the COCONUT database with a spectra
1. Enter the path to the .0 or .jdx file
2. Enter the molecular weight to filter potential target spectra
3. Default molecular weight tolerance is 0.05 daltons (named argument is mw_tol)

```text
Executes a two-stage library search using molecular weight 
and tiered recall score filtering.
Stages:
    1. Molecular Weight Filter (within specified tolerance)
    2. Recall Matching Filter (ranks results by overlap of query/target FGs)
Args:
    query_fgs (Set[str]): Set of detected functional groups.
    library_df (pd.DataFrame): DataFrame containing the chemical library database.
    mw (Optional[float]): Target query mass (typically protonated mass [M+H]+).
    mw_tol (float): Tolerance window around target mass in Daltons. Defaults to 0.05.
    top_n (int): Maximum number of records to return. Defaults to 50.
    tanimoto (float): Minimum target recall score required. Defaults to 0.5.
    search_mw (str): Column name in library_df representing the mass. Defaults to "protonated".
    nitro_rule (bool): If True, applies the nitrogen rule check to the query mass.
Returns:
    Tuple[pd.DataFrame, int]: 
        - DataFrame of top_n sorted candidates.
        - Total number of candidates remaining after Stage 1 (MW Filter).
```

In [ ]:
# agelasine
find_and_report(
    "../test_data/",               # directory to data
    "16220300001_D2.0",            # filename of data
    "C[C@@H]1CC[C@@]2([C@@H]([C@@]1(C)CC/C(=C/CN3C=[N+](C4=NC=NC(=C43)N)C)/C)CCC=C2C)C", # smiles if known for calculating hit index, not required
    422.33,                        # molecular weight 
    cls_model,                     # model to use for predicting functional groups
    search_id = "CNP0572110.6",    # COCONUT id if known, used for exact match hit index
    search_mw = "exact",           # Adduct type to search, defaults to protonated.
    tanimoto = 0.8,                # Functional group overlap score cuttoff. Only records with scores at or above will be returned
    training_data = training_data, # Reference to training data dataframe for identifying an exact match. Can be useful to verify model.
)

In [ ]:
# excluded from duplicates
find_and_report(
    "../test_data/", 
    "MTP_9_H11.0", 
    "CC(C)CCC[C@@H](C)[C@H]1CC[C@@]2(C)[C@@H]3C[C@H](OS(=O)(=O)O)[C@H]4C[C@H](OS(=O)(=O)O)[C@@H](OS(=O)(=O)O)C[C@]4(C)C3=CC[C@]12C.[Na].[Na].[Na]",
    672.23+1.007,
    cls_model,
    search_id = "CNP0572110.6",
    search_mw = "protonated",
    tanimoto = 0.8,
    training_data = training_data,
)

In [ ]:
# stilbenoids L17383_4
find_and_report(
    "../test_data/", 
    "16220300001_F11.0", 
    "C1=CC(=CC=C1/C=C/C2=C3[C@@H]([C@H](OC3=CC(=C2)O)C4=CC=C(C=C4)O)C5=CC(=C6C(C(OC6=C5)C7=CC=C(C=C7)O)C8=CC(=CC(=C8)O)O)O)O",
    681.22,
    cls_model,
    search_id = "CNP0188679.2",
    search_mw = "protonated",
    tanimoto  = 0.7,
    training_data = training_data
)

In [ ]:
# tricothecenes
find_and_report(
    "../test_data/", 
    "16220300012_B11.0", 
    "CC1=C[C@@H]2[C@]3(COC(/C=C(C)/CCO[C@@H]([C@H](O)C)/C=C/C=C\C(OC4[C@]3(C)[C@@]5([C@H](O2)C4)CO5)=O)=O)CC1",
    537.2464,
    cls_model,
    search_id = "CNP0373612.4",
    search_mw = "sodiated",
    tanimoto = 0.9,
    training_data = training_data
)

In [ ]:
# Anthraquinones
find_and_report(
    "../test_data/", 
    "16220300003_C6.0", 
    "O=C1C2=C(O)C=C(O)C=C2C(C3=C1C(O)=CC(O)=C3C(C)=O)=O",
    315.0, # 
    # 395.3636,
    cls_model,
    search_id = "CNP0171092.0",
    search_mw = "protonated",
    tanimoto  = 0.7,
    training_data = training_data
)

In [ ]:
# stilbenoids L40225_4
find_and_report(
    "../test_data/", 
    "16220300001_A8.0", 
    "C1=CC(=CC=C1/C=C/C2=C3[C@@H]([C@H](OC3=CC(=C2)O)C4=CC=C(C=C4)O)C5=CC(=C6C(C(OC6=C5)C7=CC=C(C=C7)O)C8=CC(=CC(=C8)O)O)O)O",
    681.22,
    cls_model,
    search_id = "CNP0188679.2",
    search_mw = "protonated",
    tanimoto  = 0.7,
    training_data = training_data
)

In [ ]:
# cerebrosides
find_and_report(
    "../test_data/", 
    "16220300002_B6.0", 
    "CCCCCCCCCCCCCCCCCC(O)=N[C@@]([C@@](O)([H])/C([H])=C([H])/CCCCCCCCCCC)([H])CO[C@@]1([H])[C@@](O)([H])[C@](O)([H])[C@@](O)([H])[C@@](O1)([H])CO",
    700.57, # 
    # 614.4031,
    cls_model,
    search_id = None,
    search_mw = "protonated",
    tanimoto  = 0.2,
    training_data = training_data
)

In [ ]:
# Perfragilin B
find_and_report(
    "../test_data/", 
    "16220300012_A7.0", 
    "CN1C=C(C2=CC1=O)C(C(SC)=C(SC)C2=O)=O",
    # 315.26, # 
    282.0260,
    cls_model,
    search_id = "CNP0157927.0",
    search_mw = "protonated",
    tanimoto  = 0.8,
    training_data = training_data
)

In [ ]:
# Diketopiperazine class - Dimethylgliotoxin,Bis(Dethio)Bis(Methylthio)Gliotoxin H210459_4
find_and_report(
    "../test_data/", 
    "16220300012_D7.0", 
    "O=C1N(C)C(SC)(CO)C(N2C1(SC)CC3=CC=CC(O)C23[H])=O",
    # 315.26, # 
    357.0940,
    cls_model,
    search_id = "CNP0299713.1",
    search_mw = "protonated",
    tanimoto  = 0.8,
    training_data = training_data
)

In [ ]:
# rutin from NIST jdx file
find_and_report(
    "../test_data/", 
    "B6005093-IR.jdx", 
    "C[C@@H]1O[C@@H](OC[C@H]2O[C@@H](OC3=C(C4=CC=C(O)C(O)=C4)OC4=CC(O)=CC(O)=C4C3=O)[C@H](O)[C@@H](O)[C@@H]2O)[C@H](O)[C@H](O)[C@H]1O",
    # 315.26, # 
    611.160661,
    cls_model,
    search_id = "CNP0268715.1",
    search_mw = "protonated",
    tanimoto  = 0.8,
    training_data = training_data,
)